# Inter Rater Agreement

Compares the second rater's independent verdicts against the first rater's and reports the
agreement statistics specified in advance in `Drafts/second_rater_protocol.md`.

**Reads** `Handover/scoring_form.xlsx` (rater 2) and `Data/Tracker/exports/scoring_long.csv`
(rater 1). **Writes** `Data/Tracker/exports/agreement_summary.csv` and
`Data/Tracker/exports/disagreements.csv`.

This notebook is read only with respect to the appraisal record. It never edits the tracker.

### Two rules fixed before any data existed

**Computed on the independent ratings only.** Run this before you and the rater discuss
anything. Reconciling first and measuring afterwards produces a number that means nothing, and
the figure reported in the manuscript is the one this notebook produces on first run.

**Never recomputed after reconciliation.** If verdicts change during reconciliation, the
tracker changes and the appraisal results change with it, but the agreement statistic does not.
It describes how two people independently applied the instrument, which is a fact about a moment
that reconciliation destroys.

Run it once. Keep the output.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT    = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
# Which completed rating to compare against rater 1. Handover/scoring_form.xlsx is the
# BLANK template and is never read here. Point this at a human rating when one exists;
# D12 makes a human result primary automatically, whichever agreement figure is higher.
FORM    = ROOT / "Data" / "Reliability" / "rating_model_run1.xlsx"
LONG    = ROOT / "Data" / "Tracker" / "exports" / "scoring_long.csv"
EXPORTS = ROOT / "Data" / "Tracker" / "exports"

CATS   = ["Yes", "No", "Unclear"]        # substantive categories
CATS4  = CATS + ["N/A"]

# --- rater 1, from the tracker exports -----------------------------------
r1 = pd.read_csv(LONG, keep_default_na=False)[["paper_id", "citation", "item_no", "item", "verdict"]]
r1 = r1.rename(columns={"verdict": "r1"})

# --- rater 2, from the handover form -------------------------------------
# keep_default_na=False is REQUIRED. Without it pandas parses the literal verdict "N/A" as a
# missing value, silently turning every legitimate N/A verdict into an unscored blank. The same
# trap applies to scoring_long.csv above. It cost a debugging cycle; do not remove it.
r2 = pd.read_excel(FORM, sheet_name="Scoring", keep_default_na=False)
r2 = r2.rename(columns={"Paper": "paper_id", "Item #": "item_no",
                        "VERDICT": "r2", "CONFIDENCE": "conf", "NOTE": "note"})
r2 = r2[["paper_id", "item_no", "r2", "conf", "note"]]
r2["r2"] = r2["r2"].astype(str).str.strip().replace({"nan": "", "None": ""})
assert (r2["r2"] != "nan").all(), "N/A verdicts were coerced to NaN on read"

SAMPLE = sorted(r2["paper_id"].unique())
n_filled = int((r2["r2"] != "").sum())

print(f"rater 2 form   {FORM.relative_to(ROOT)}")
print(f"papers in sample   {SAMPLE}")
print(f"cells in form      {len(r2)}")
print(f"cells scored       {n_filled}")

READY = n_filled == len(r2)
if not READY:
    print()
    print("=" * 74)
    print("  FORM NOT COMPLETE. Statistics below will not be computed.")
    print(f"  {len(r2) - n_filled} of {len(r2)} cells are still blank.")
    print("  This is the expected state until the rater returns the form.")
    print("=" * 74)
else:
    bad = sorted(set(r2["r2"]) - set(CATS4))
    assert not bad, f"unrecognised verdicts in the form: {bad}"


## The statistics, and why each one is here

No single coefficient is trustworthy on this data, so three are reported together.

| Statistic | What it does | Why it alone is not enough |
| --- | --- | --- |
| **Raw percent agreement** | The proportion of cells where both raters wrote the same thing | Ignores agreement that would happen by chance, so it flatters |
| **Cohen's kappa** | Corrects for chance agreement given each rater's marginal habits | Collapses toward zero when one category dominates, even under near perfect agreement |
| **PABAK** and **Gwet's AC1** | Correct for chance without being destabilised by skewed prevalence | Less familiar to readers, so they are reported as companions rather than as the headline |

The prevalence problem is real here rather than theoretical. Across the full appraisal, 126 of
184 scoreable cells are `Yes`. At that skew Cohen's kappa is pushed downward even when two
raters agree nearly everywhere, which is the well known kappa paradox. Reporting kappa alone
would understate reliability. Reporting percent agreement alone would overstate it.

**Per item kappa is deliberately not computed for near degenerate items.** Where one category
holds every cell, kappa is undefined or wildly unstable even under perfect agreement. Those
items get percent agreement and a note saying why.

In [ ]:
def confusion(a, b, cats):
    """Square contingency table of two raters' labels."""
    idx = {c: i for i, c in enumerate(cats)}
    m = np.zeros((len(cats), len(cats)), dtype=float)
    for x, y in zip(a, b):
        m[idx[x], idx[y]] += 1
    return m


def percent_agreement(a, b):
    a, b = np.asarray(a), np.asarray(b)
    return float((a == b).mean()) if len(a) else np.nan


def cohen_kappa(a, b, cats):
    """Cohen's unweighted kappa, with its asymptotic standard error."""
    m = confusion(a, b, cats)
    n = m.sum()
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan
    po = np.trace(m) / n
    r, c = m.sum(axis=1) / n, m.sum(axis=0) / n
    pe = float((r * c).sum())
    if np.isclose(pe, 1.0):
        return np.nan, po, pe, np.nan          # undefined: one category holds everything
    k = (po - pe) / (1 - pe)
    se = np.sqrt(po * (1 - po) / (n * (1 - pe) ** 2))   # simple asymptotic SE
    return float(k), float(po), pe, float(se)


def pabak(po, k_cats):
    """Prevalence adjusted bias adjusted kappa, generalised to k categories."""
    return (k_cats * po - 1) / (k_cats - 1)


def gwet_ac1(a, b, cats):
    """Gwet's AC1. Chance agreement estimated without the prevalence instability."""
    m = confusion(a, b, cats)
    n = m.sum()
    if n == 0:
        return np.nan
    po = np.trace(m) / n
    q = len(cats)
    pi = (m.sum(axis=1) / n + m.sum(axis=0) / n) / 2
    pe = float((pi * (1 - pi)).sum()) / (q - 1)
    return float((po - pe) / (1 - pe)) if not np.isclose(pe, 1.0) else np.nan


def landis_koch(k):
    """Conventional interpretation bands. A convention, not a threshold of truth."""
    if not np.isfinite(k):
        return "undefined"
    if k < 0:    return "poor"
    if k < 0.21: return "slight"
    if k < 0.41: return "fair"
    if k < 0.61: return "moderate"
    if k < 0.81: return "substantial"
    return "almost perfect"


# --- self test on a textbook example, so a formula error cannot pass silently ---
_a = ["Yes"] * 20 + ["Yes"] * 5  + ["No"] * 10 + ["No"] * 15
_b = ["Yes"] * 20 + ["No"]  * 5  + ["Yes"] * 10 + ["No"] * 15
_k, _po, _pe, _se = cohen_kappa(_a, _b, ["Yes", "No"])
assert abs(_po - 0.70) < 1e-9, _po
assert abs(_pe - 0.50) < 1e-9, _pe
assert abs(_k  - 0.40) < 1e-9, _k          # (0.70 - 0.50) / (1 - 0.50)
assert abs(pabak(_po, 2) - 0.40) < 1e-9
assert abs(percent_agreement(_a, _b) - 0.70) < 1e-9
_perfect = ["Yes"] * 14
assert not np.isfinite(cohen_kappa(_perfect, _perfect, ["Yes", "No"])[0])   # degenerate -> undefined
assert abs(cohen_kappa(["Yes","No"], ["Yes","No"], ["Yes","No"])[0] - 1.0) < 1e-9
print("agreement functions loaded and self tested")
print("  textbook check: Po = 0.70, Pe = 0.50, kappa = 0.40  OK")


## Overall agreement

In [ ]:
if not READY:
    print("Form incomplete. Nothing to compute yet.")
else:
    m = r1.merge(r2, on=["paper_id", "item_no"], how="inner", validate="one_to_one")
    assert len(m) == len(r2), f"merge lost rows: {len(m)} vs {len(r2)}"

    both_scoreable = m[(m.r1 != "N/A") & (m.r2 != "N/A")]
    k3, po3, pe3, se3 = cohen_kappa(both_scoreable.r1, both_scoreable.r2, CATS)
    k4, po4, pe4, se4 = cohen_kappa(m.r1, m.r2, CATS4)

    rows = []
    def add(label, stat, value, n, note=""):
        rows.append({"scope": label, "statistic": stat, "value": value, "n_cells": n, "note": note})

    print("OVERALL AGREEMENT")
    print("=" * 78)
    print(f"  cells compared                      {len(m)}")
    print(f"  cells scoreable by BOTH raters      {len(both_scoreable)}")
    print()
    print("  --- substantive categories only (Yes / No / Unclear) ---")
    print(f"  raw percent agreement               {po3*100:5.1f}%")
    print(f"  Cohen's kappa                       {k3:5.3f}   ({landis_koch(k3)})")
    if np.isfinite(se3):
        print(f"     95% CI (asymptotic)              {k3-1.96*se3:5.3f} to {k3+1.96*se3:5.3f}")
    print(f"  PABAK                               {pabak(po3, 3):5.3f}")
    print(f"  Gwet's AC1                          {gwet_ac1(both_scoreable.r1, both_scoreable.r2, CATS):5.3f}")
    print()
    print("  --- all four categories, N/A treated as a category ---")
    print(f"  raw percent agreement               {po4*100:5.1f}%")
    print(f"  Cohen's kappa                       {k4:5.3f}   ({landis_koch(k4)})")
    print(f"  PABAK                               {pabak(po4, 4):5.3f}")
    print(f"  Gwet's AC1                          {gwet_ac1(m.r1, m.r2, CATS4):5.3f}")
    print("=" * 78)
    print()
    print("  The asymptotic interval assumes cells are independent. They are not, since")
    print(f"  they are clustered within {m.paper_id.nunique()} papers. Read it as optimistic.")

    add("substantive", "percent_agreement", po3, len(both_scoreable))
    add("substantive", "cohen_kappa", k3, len(both_scoreable), landis_koch(k3))
    add("substantive", "kappa_se", se3, len(both_scoreable))
    add("substantive", "pabak", pabak(po3, 3), len(both_scoreable))
    add("substantive", "gwet_ac1", gwet_ac1(both_scoreable.r1, both_scoreable.r2, CATS), len(both_scoreable))
    add("all_four", "percent_agreement", po4, len(m))
    add("all_four", "cohen_kappa", k4, len(m), landis_koch(k4))
    add("all_four", "pabak", pabak(po4, 4), len(m))


## Per item breakdown

Where disagreement concentrates matters more than the overall figure. An item that two careful
readers cannot apply the same way is a defect in the instrument, and that is a reportable
finding in its own right.

In [ ]:
if not READY:
    print("Form incomplete.")
else:
    per = []
    for i, g in m.groupby("item_no"):
        both = g[(g.r1 != "N/A") & (g.r2 != "N/A")]
        pa = percent_agreement(g.r1, g.r2)
        distinct = len(set(both.r1) | set(both.r2))
        if len(both) >= 2 and distinct >= 2:
            ki, _, _, _ = cohen_kappa(both.r1, both.r2, CATS)
            reason = ""
        else:
            ki, reason = np.nan, "near degenerate, kappa not computed"
        per.append({"item_no": i, "item": g.item.iloc[0], "n": len(g),
                    "n_both_scoreable": len(both), "percent_agreement": pa,
                    "cohen_kappa": ki, "note": reason,
                    "n_disagreements": int((g.r1 != g.r2).sum())})
    per = pd.DataFrame(per).sort_values(["n_disagreements", "item_no"], ascending=[False, True])

    print("PER ITEM AGREEMENT")
    print("=" * 88)
    print(f"{'#':>2}  {'item':<26}{'n':>3}{'agree':>8}{'kappa':>8}   disagreements / note")
    print("-" * 88)
    for _, r in per.iterrows():
        kt = f"{r.cohen_kappa:6.3f}" if np.isfinite(r.cohen_kappa) else "     -"
        tail = f"{int(r.n_disagreements)}" + (f"   {r['note']}" if r["note"] else "")
        print(f"{int(r.item_no):>2}  {r['item']:<26}{int(r.n):>3}{r.percent_agreement*100:7.0f}%{kt}   {tail}")
    print("=" * 88)
    worst = per.iloc[0]
    if worst.n_disagreements > 0:
        print(f"\n  Most contested item: {int(worst.item_no)} {worst['item']}, "


## Where the `N/A` rule was applied differently

This is a direct test of whether the item class rule is comprehensible to someone who did not
invent it. Disagreement here is not a disagreement about a paper. It is a disagreement about the
instrument.

In [ ]:
if not READY:
    print("Form incomplete.")
else:
    r1_na, r2_na = (m.r1 == "N/A"), (m.r2 == "N/A")
    both_na  = int((r1_na & r2_na).sum())
    only_r1  = int((r1_na & ~r2_na).sum())
    only_r2  = int((~r1_na & r2_na).sum())
    neither  = int((~r1_na & ~r2_na).sum())
    conc = (both_na + neither) / len(m)

    print("N/A CONCORDANCE")
    print("=" * 60)
    print(f"  both marked N/A            {both_na:>3}")
    print(f"  rater 1 only               {only_r1:>3}")
    print(f"  rater 2 only               {only_r2:>3}")
    print(f"  neither                    {neither:>3}")
    print(f"  concordance                {conc*100:.1f}%")
    print("=" * 60)

    forbidden = m[(m.item_no.isin([2, 9])) & ((m.r1 == "N/A") | (m.r2 == "N/A"))]
    if len(forbidden):
        print("\n  RULE VIOLATION: N/A used on a class A item (2 or 9), where it is never permitted.")
        print(forbidden[["paper_id", "item_no", "r1", "r2"]].to_string(index=False))
    else:


## Did disagreement land where the rater felt unsure

The confidence column exists so this question can be asked. If disagreements concentrate in
cells the rater already flagged `Low`, the instrument is behaving sensibly and the uncertainty
is visible from inside. If they are spread evenly across confidence levels, that is worse news,
because it means neither rater could tell which of their judgments were shaky.

In [ ]:
if not READY:
    print("Form incomplete.")
else:
    m["agreed"] = m.r1 == m.r2
    if m.conf.notna().any():
        tab = (m.groupby(m.conf.fillna("(blank)"))
                 .agg(cells=("agreed", "size"), disagreements=("agreed", lambda s: int((~s).sum())))
                 .assign(rate=lambda d: d.disagreements / d.cells))
        order = [c for c in ["High", "Medium", "Low", "(blank)"] if c in tab.index]
        tab = tab.loc[order]
        print("DISAGREEMENT BY RATER 2 CONFIDENCE")
        print("=" * 56)
        print(f"{'confidence':<12}{'cells':>7}{'disagree':>10}{'rate':>9}")
        print("-" * 56)
        for c, r in tab.iterrows():
            print(f"{c:<12}{int(r.cells):>7}{int(r.disagreements):>10}{r.rate*100:8.0f}%")
        print("=" * 56)
    else:


## The disagreement log

Every cell where the two raters differ, written out for reconciliation. This is the working
document for the conversation that follows, and it is also the audit trail showing that
reconciliation happened after measurement rather than before it.

In [ ]:
if not READY:
    print("Form incomplete. No log written.")
else:
    d = m[m.r1 != m.r2].copy()
    d = d[["paper_id", "citation", "item_no", "item", "r1", "r2", "conf", "note"]]
    d.columns = ["paper_id", "citation", "item_no", "item",
                 "rater1_verdict", "rater2_verdict", "rater2_confidence", "rater2_note"]
    d["agreed_resolution"] = ""          # to fill during reconciliation
    d["resolution_rationale"] = ""
    d["tracker_updated"] = ""
    d = d.sort_values(["item_no", "paper_id"])
    d.to_csv(EXPORTS / "disagreements.csv", index=False)

    summary = pd.DataFrame(rows)
    summary.to_csv(EXPORTS / "agreement_summary.csv", index=False)

    print(f"{len(d)} disagreements of {len(m)} cells ({len(d)/len(m)*100:.1f}%)")
    print(f"wrote  {(EXPORTS / 'disagreements.csv').relative_to(ROOT)}")
    print(f"wrote  {(EXPORTS / 'agreement_summary.csv').relative_to(ROOT)}")
    if len(d):
        print()
        print(d[["paper_id", "item_no", "item", "rater1_verdict",


## Reporting this

Quote all three coefficients together, never kappa alone.

State the sample explicitly. Five papers of fifteen, sixty five cells, dual rated, with the
papers chosen by seeded random draw stratified by domain before the ratings began.

Call the confidence interval approximate. Cells are clustered within five papers and the
asymptotic interval does not account for that.

If kappa lands below about 0.40, resist reading it as a failure by either rater. On an
instrument this skewed toward `Yes` it is at least as likely to be a property of the coefficient
or of ambiguous item wording. The per item table will usually show which, because a genuine
wording problem concentrates in one or two items rather than spreading evenly.

Report the items where the two raters diverged most, and say what the ambiguity was. That is
more useful to a reader than the coefficient, and it is the part that tells them whether they
could apply this checklist themselves.

In [ ]:
print("STATUS")
print("=" * 66)
print(f"  form            {'complete' if READY else 'NOT YET COMPLETE'}")
print(f"  cells           {n_filled} of {len(r2)} scored")
for f in ["agreement_summary.csv", "disagreements.csv"]:
    p = EXPORTS / f
    print(f"  {'OK ' if p.exists() else '-- '}{f}")
print("=" * 66)
print("\nALL CELLS EXECUTED CLEANLY")
